# 👁️ Diabetic Retinopathy Classifier — Training Notebook

**Model**: EfficientNet-B0 (Transfer Learning + Fine-tuning)  
**Framework**: TensorFlow / Keras  
**Dataset**: [APTOS 2019 Blindness Detection](https://www.kaggle.com/competitions/aptos2019-blindness-detection/data)  
**Runtime**: GPU (T4 recommended)

---

### Pipeline Summary
1. Mount Google Drive & install dependencies  
2. Load & explore the APTOS 2019 dataset  
3. Preprocess retinal images (CLAHE, local mean subtraction)  
4. Build EfficientNet-B0 model  
5. Phase 1 — Feature extraction (frozen base, 15 epochs)  
6. Phase 2 — Fine-tuning (unfreeze last 20 layers, 15 more epochs)  
7. Evaluate & plot results  
8. Save `.h5` model to Drive  

In [ ]:
# ── 0. Colab GPU Check ────────────────────────────────────────────────────────
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── 1. Mount Drive & clone repo ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Clone the project (update URL to your repo)
!git clone https://github.com/YOUR_USERNAME/diabetic-retinopathy-classifier.git /content/dr_project
%cd /content/dr_project

!pip install -r requirements.txt -q

In [ ]:
# ── 2. Download APTOS 2019 from Kaggle ────────────────────────────────────────
# Upload your kaggle.json first:
from google.colab import files
files.upload()  # upload kaggle.json

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions download -c aptos2019-blindness-detection -p /content/data
!unzip -q /content/data/aptos2019-blindness-detection.zip -d /content/data

In [ ]:
# ── 3. Explore dataset ────────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('/content/data/train.csv')
print(df.head())
print('\nClass distribution:')
print(df['diagnosis'].value_counts().sort_index())

# Plot class distribution
labels = ['No DR','Mild','Moderate','Severe','Proliferative']
counts = df['diagnosis'].value_counts().sort_index()
colors = ['#2ecc71','#f1c40f','#e67e22','#e74c3c','#8e44ad']
plt.figure(figsize=(8,4))
plt.bar(labels, counts, color=colors, edgecolor='none')
plt.title('APTOS 2019 — Class Distribution', fontsize=13)
plt.ylabel('Number of Images')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Sample retinal images ──────────────────────────────────────────────────
import cv2, os

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for grade in range(5):
    sample = df[df['diagnosis'] == grade].iloc[0]['id_code']
    img_path = f'/content/data/train_images/{sample}.png'
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    axes[grade].imshow(img)
    axes[grade].set_title(f'Grade {grade}', fontsize=10)
    axes[grade].axis('off')
plt.suptitle('Sample Retinal Fundus Images by Grade', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Build model ────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/dr_project')
from model.model_builder import build_model, model_summary

model = build_model(freeze_base=True)
model_summary(model)

In [ ]:
# ── 6. Run full training pipeline ────────────────────────────────────────────
!python model/train.py \
    --data_dir /content/data \
    --csv_path /content/data/train.csv \
    --output_dir /content/drive/MyDrive/DR_Model \
    --epochs 30 \
    --batch_size 32

In [ ]:
# ── 7. Plot training history ──────────────────────────────────────────────────
from PIL import Image as PILImage
img = PILImage.open('/content/drive/MyDrive/DR_Model/training_history.png')
plt.figure(figsize=(12,4))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# ── 8. Copy model to project directory ───────────────────────────────────────
import shutil
shutil.copy(
    '/content/drive/MyDrive/DR_Model/dr_efficientnet_b0.h5',
    '/content/dr_project/model/dr_efficientnet_b0.h5'
)
print('✓ Model copied to project folder')
print('You can now run the Streamlit app with:')
print('  streamlit run app/main.py')